In [ ]:
from datasets import load_from_disk
import logging
import json
import requests
import os, time
import random
from typing import List, TypeVar, Type, Optional
from openai import AzureOpenAI, OpenAIError
from pydantic import BaseModel, ValidationError
from pydantic import ValidationError
from json import JSONDecodeError
import sys

sys.path.append('../')
from models.classes_pydantic import RegistroPedido, ResumoPedido



In [2]:
ds = load_from_disk('../etl/datasets/ds_lai_pedidos')

In [3]:
ds[3]

{'IdPedido': 10141818,
 'ProtocoloPedido': '23546000005202618',
 'Esfera': 'Federal',
 'OrgaoDestinatario': 'UFPI – Fundação Universidade Federal do Piauí',
 'Situacao': 'Concluída',
 'DataRegistro': '01/01/2026',
 'ResumoSolicitacao': 'Informações sobre os TEDs efetivados pela UFPI de 2021 até 2025',
 'DetalhamentoSolicitacao': 'I. Contexto Considerando a relevância da transparência na execução de instrumentos de descentralização de créditos orçamentários, solicita-se, nos termos da Lei nº 12.527/2011, informações detalhadas sobre os Termos de Execução Descentralizada (TEDs) efetivados pela UFPI no exercício corrente.  II. Informações Requeridas TEDs efetivados  Quais Termos de Execução Descentralizada (TEDs) foram efetivados pela UFPI nos anos de 2021 até 2025?  Execução orçamentária  Quais desses TEDs foram empenhados e quais foram liquidados?  Informar os valores correspondentes a cada fase da execução (empenho, liquidação e pagamento, se houver).  Registro processual  Qual o númer

In [4]:
RegistroPedido(protocoloPedido=ds[3]['ProtocoloPedido'],
                orgaoDestinatario=ds[3]['OrgaoDestinatario'],
                resumoSolicitacao=ds[3]['ResumoSolicitacao'],
                detalhamentoSolicitacao=ds[3]['DetalhamentoSolicitacao'],
                assuntoPedido=ds[3]['AssuntoPedido'],
                subAssuntoPedido=ds[3]['SubAssuntoPedido'],
                tag=ds[3]['Tag'],
                resposta=ds[3]['Resposta'],
                decisao=ds[3]['Decisao'],
                detalhamentoDecisao=ds[3]['DetalhamentoDecisao'],
                motivoNegativaAcesso=ds[3]['MotivoNegativaAcesso'],
               )

RegistroPedido(protocoloPedido='23546000005202618', orgaoDestinatario='UFPI – Fundação Universidade Federal do Piauí', resumoSolicitacao='Informações sobre os TEDs efetivados pela UFPI de 2021 até 2025', detalhamentoSolicitacao='I. Contexto Considerando a relevância da transparência na execução de instrumentos de descentralização de créditos orçamentários, solicita-se, nos termos da Lei nº 12.527/2011, informações detalhadas sobre os Termos de Execução Descentralizada (TEDs) efetivados pela UFPI no exercício corrente.  II. Informações Requeridas TEDs efetivados  Quais Termos de Execução Descentralizada (TEDs) foram efetivados pela UFPI nos anos de 2021 até 2025?  Execução orçamentária  Quais desses TEDs foram empenhados e quais foram liquidados?  Informar os valores correspondentes a cada fase da execução (empenho, liquidação e pagamento, se houver).  Registro processual  Qual o número do processo no SIPAC vinculado a cada TED efetivado?  Solicita-se a disponibilização de cópia dos doc

In [ ]:
def call_llm_with_retry_and_pydantic_response(
        system_prompt: str,
        user_prompt: str,
        formato_pydantic: Type[T] = ResumoPedido,
        client: AzureOpenAI = None,
        modelo_llm: Optional[str] = None,
        max_retries: int = 10,
        initial_backoff: float = 1.0,
        backoff_factor: float = 1.5,
    ) -> T:
        
        if client is None:
            client = AzureOpenAI(api_key=self.token_prop, 
                                    api_version="2024-10-21", 
                                    azure_endpoint=self.lia_url_prop) 


        if modelo_llm is None:
            modelo_llm = config_prop.modelo_nome_e_versao

        instrucao_do_formato = (
            "\nResponda EXCLUSIVAMENTE com JSON válido seguindo o schema abaixo.\n\n"
            f"{json.dumps(formato_pydantic.model_json_schema(), indent=2)}" 
        )

        user_prompt = user_prompt + instrucao_do_formato

        attempt = 0
        current_backoff = initial_backoff

        while attempt < max_retries:
            try:
                response= client.chat.completions.create( 
                    model=modelo_llm,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    response_format={"type": "json_object"},
                    temperature = 0.01
                )

                # Retorna JSON parseado
                parsed = response.choices[0].message.content

                if parsed is None:
                    raise ValueError("O LLM retornou `None` em vez de JSON válido")
                
                parsed_dict = json.loads(parsed)  # converte string JSON em dict
                #print(parsed_dict)

                formato_ok = formato_pydantic.model_validate(parsed_dict)
                #print(formato_ok)
                
                return formato_ok

            except ValidationError as e:
                raise ValueError(f"Schema inválido: {e}")

            except JSONDecodeError as e:
                raise ValueError(f"JSON inválido: {e}")

            except OpenAIError as e:
                status_code = getattr(e, "status_code", None)
                if status_code == 403:
                    print(f"Erro de permissão (403): {str(e)}")
                    raise
                print(f"Tentativa {attempt + 1} falhou (OpenAIError): {str(e)}")

            except Exception as e:
                print(f"Tentativa {attempt + 1} falhou (Erro local): {str(e)}")

            # Retry com backoff exponencial + jitter
            attempt += 1
            if attempt < max_retries:
                jitter = random.uniform(0.9, 1.1)
                sleep_time = current_backoff * jitter
                print(f"Aguardando {sleep_time:.2f}s antes da próxima tentativa...")

                time.sleep(sleep_time)

                current_backoff *= backoff_factor
            else:
                raise ConnectionRefusedError(f"Falha após {max_retries} tentativas.")
            
        raise ConnectionRefusedError("Limite de tentativas atingido sem resposta válida da LLM.")
